# 🏥 Family Health Guardian: Unified Hybrid Machine Learning Suite
### Train 3 Advanced Hybrid Models with Kaggle API & Google Colab

This master notebook downloads clinical datasets from Kaggle using your `kaggle.json`, trains **3 distinct Hybrid ML architectures**, benchmarks them against baseline models, and downloads all deployment-ready model artifacts to your computer.

---

### 📋 The 3 Hybrid Models in This Suite:
1. **Model 1: Two-Tier Stacking Super Learner (Diabetes Risk Prediction)**
   - *Level 0 Base Learners*: Random Forest + Gradient Boosting + Support Vector Machine
   - *Level 1 Meta-Learner*: L2 Regularized Logistic Regression
   - *Dataset*: `iammustafatz/diabetes-prediction-dataset` (100k records)

2. **Model 2: Weighted Soft-Voting Blended Ensemble (Hypertension Risk)**
   - *Ensemble Components*: Balanced Random Forest (40%) + ExtraTrees (35%) + HistGradientBoosting (25%)
   - *Optimization*: Cost-sensitive recall optimization for clinical screening
   - *Dataset*: `iammustafatz/diabetes-prediction-dataset` (Cardiovascular subset)

3. **Model 3: Multi-Disease Diagnostic Symptom Classifier (41 Diseases & 132 Symptoms)**
   - *Architecture*: Multi-class probability estimator with top-3 differential diagnosis
   - *Dataset*: `itachi9604/disease-symptom-description-dataset`
   - *Includes*: Integrated descriptions, symptom severities, and 4-step medical precautions


## ⚙️ Step 1: Install Dependencies

In [ ]:
!pip install -q kaggle scikit-learn pandas numpy joblib matplotlib seaborn
print("✅ All dependencies installed successfully!")

## 🔑 Step 2: Upload `kaggle.json` & Configure Kaggle API
Click **Choose Files** below and select your `kaggle.json` token file from your computer.

In [ ]:
import os
from google.colab import files

print("👉 Please select and upload your 'kaggle.json' file:")
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("\n✅ kaggle.json configured successfully!")
else:
    print("\n⚠️ 'kaggle.json' was not uploaded. Please run this cell again and select kaggle.json.")

## 📥 Step 3: Automatically Download Both Datasets from Kaggle
Downloads:
1. `iammustafatz/diabetes-prediction-dataset` (Metabolic & cardiovascular patient records)
2. `itachi9604/disease-symptom-description-dataset` (41 diseases & symptoms)

In [ ]:
!mkdir -p raw_datasets

print("Downloading Dataset 1 (Diabetes & Cardiovascular)...\n")
!kaggle datasets download -d iammustafatz/diabetes-prediction-dataset --path raw_datasets --unzip

print("\nDownloading Dataset 2 (Multi-Disease Symptoms)...\n")
!kaggle datasets download -d itachi9604/disease-symptom-description-dataset --path raw_datasets/disease_symptoms --unzip

print("\n📁 Downloaded files:")
!ls -lh raw_datasets
!ls -lh raw_datasets/disease_symptoms

================================================================================
# 🧠 MODEL 1: Two-Tier Stacking Super Learner for Diabetes Risk
================================================================================
Combines Random Forest, Gradient Boosting, and Support Vector Classifier via a Logistic Regression Meta-Learner.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, classification_report
import time

# 1. Load and clean
df_diab = pd.read_csv("raw_datasets/diabetes_prediction_dataset.csv").drop_duplicates().reset_index(drop=True)
print(f"Diabetes Dataset Shape (Deduplicated): {df_diab.shape}")

# 2. One-hot encoding with dummy trap prevention (drop_first=True)
gender_dummies = pd.get_dummies(df_diab['gender'], prefix='gender', drop_first=True, dtype=int)
smoking_dummies = pd.get_dummies(df_diab['smoking_history'], prefix='smoking_history', drop_first=True, dtype=int)

for col in ['gender_Male', 'gender_Other']:
    if col not in gender_dummies.columns:
        gender_dummies[col] = 0

for col in ['smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current']:
    if col not in smoking_dummies.columns:
        smoking_dummies[col] = 0

feature_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']
X_diab = pd.concat([df_diab[feature_cols], gender_dummies[['gender_Male', 'gender_Other']], smoking_dummies[['smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current']]], axis=1)
y_diab = df_diab['diabetes'].astype(int)

# Exact 13 features expected by production API
expected_diab_features = [
    'age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level',
    'gender_Male', 'gender_Other',
    'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'
]
X_diab = X_diab[expected_diab_features]

# 3. Stratified split (80% train, 20% test)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_diab, y_diab, test_size=0.20, random_state=42, stratify=y_diab)

# 4. Define Level 0 Base Learners
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.1, random_state=42)),
    ('lr_base', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
]

# 5. Define Level 1 Meta-Learner
meta_learner = LogisticRegression(penalty='l2', C=1.0, random_state=42)

print("Training Stacking Super Learner for Diabetes...")
t0 = time.time()
diab_stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)
diab_stacking_model.fit(X_train_d, y_train_d)
print(f"✅ Stacking model trained in {time.time() - t0:.2f}s")

# 6. Evaluate
y_pred_d = diab_stacking_model.predict(X_test_d)
y_prob_d = diab_stacking_model.predict_proba(X_test_d)[:, 1]

diab_metrics = {
    "Model": "Hybrid Stacking Classifier (RF + GB + LR)",
    "Accuracy": accuracy_score(y_test_d, y_pred_d),
    "Recall": recall_score(y_test_d, y_pred_d),
    "Precision": precision_score(y_test_d, y_pred_d),
    "F1-Score": f1_score(y_test_d, y_pred_d),
    "ROC-AUC": roc_auc_score(y_test_d, y_prob_d)
}

print("\n--- Model 1 Evaluation Results ---")
for k, v in diab_metrics.items():
    if isinstance(v, float):
        print(f"{k:15}: {v*100:.2f}%")
    else:
        print(f"{k:15}: {v}")

================================================================================
# 💓 MODEL 2: Weighted Soft-Voting Blended Ensemble for Hypertension
================================================================================
Combines Balanced Random Forest, ExtraTrees, and HistGradientBoosting with probability blending to maximize positive screening recall.

In [ ]:
from sklearn.ensemble import VotingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

# Construct Hypertension dataset
feature_cols_htn = ['age', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
X_htn = pd.concat([df_diab[feature_cols_htn], gender_dummies[['gender_Male', 'gender_Other']], smoking_dummies[['smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current']]], axis=1)
y_htn = df_diab['hypertension'].astype(int)

expected_htn_features = [
    'age', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes',
    'gender_Male', 'gender_Other',
    'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'
]
X_htn = X_htn[expected_htn_features]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_htn, y_htn, test_size=0.20, random_state=42, stratify=y_htn)

# Ensemble components
rf_htn = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
et_htn = ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
hgb_htn = HistGradientBoostingClassifier(class_weight='balanced', max_iter=100, random_state=42)

print("Training Weighted Soft-Voting Blended Ensemble for Hypertension...")
t0 = time.time()
htn_voting_model = VotingClassifier(
    estimators=[('rf', rf_htn), ('et', et_htn), ('hgb', hgb_htn)],
    voting='soft',
    weights=[0.40, 0.35, 0.25],
    n_jobs=-1
)
htn_voting_model.fit(X_train_h, y_train_h)
print(f"✅ Voting model trained in {time.time() - t0:.2f}s")

# Evaluate
y_pred_h = htn_voting_model.predict(X_test_h)
y_prob_h = htn_voting_model.predict_proba(X_test_h)[:, 1]

htn_metrics = {
    "Model": "Weighted Soft-Voting Ensemble (RF 40% + ET 35% + HGB 25%)",
    "Accuracy": accuracy_score(y_test_h, y_pred_h),
    "Recall": recall_score(y_test_h, y_pred_h),
    "Precision": precision_score(y_test_h, y_pred_h),
    "F1-Score": f1_score(y_test_h, y_pred_h),
    "ROC-AUC": roc_auc_score(y_test_h, y_prob_h)
}

print("\n--- Model 2 Evaluation Results ---")
for k, v in htn_metrics.items():
    if isinstance(v, float):
        print(f"{k:15}: {v*100:.2f}%")
    else:
        print(f"{k:15}: {v}")

================================================================================
# 🩺 MODEL 3: Multi-Disease Diagnostic Classifier (41 Diseases & 132 Symptoms)
================================================================================
Trains an ensemble model across 41 conditions and integrates disease descriptions, precautions, and symptom severities.

In [ ]:
data_sym_dir = Path("raw_datasets/disease_symptoms")
df_s_dataset = pd.read_csv(data_sym_dir / "dataset.csv")
df_s_desc = pd.read_csv(data_sym_dir / "symptom_Description.csv")
df_s_prec = pd.read_csv(data_sym_dir / "symptom_precaution.csv")

# Clean symptoms
sym_cols = [c for c in df_s_dataset.columns if c.startswith('Symptom')]
all_symptoms = set()
for c in sym_cols:
    clean_series = df_s_dataset[c].dropna().astype(str).str.strip().str.replace(' ', '_').str.lower()
    all_symptoms.update(clean_series.unique())

unique_symptoms = sorted([s for s in all_symptoms if s and s != 'nan'])
print(f"Identified {len(unique_symptoms)} unique clinical symptoms across {df_s_dataset['Disease'].nunique()} diseases.")

# Construct binary feature matrix
records = []
disease_targets = []
for _, row in df_s_dataset.iterrows():
    d_name = str(row['Disease']).strip()
    active_syms = set()
    for c in sym_cols:
        val = row[c]
        if pd.notna(val):
            active_syms.add(str(val).strip().replace(' ', '_').lower())
    record = {s: (1 if s in active_syms else 0) for s in unique_symptoms}
    records.append(record)
    disease_targets.append(d_name)

X_sym = pd.DataFrame(records)
y_sym = pd.Series(disease_targets)

# Stratified split
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_sym, y_sym, test_size=0.20, random_state=42, stratify=y_sym)

# Train Multi-Class Random Forest Model
print("Training Multi-Disease Diagnostic Classifier...")
t0 = time.time()
multi_disease_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
multi_disease_model.fit(X_train_s, y_train_s)
print(f"✅ Multi-Disease model trained in {time.time() - t0:.2f}s")

y_pred_s = multi_disease_model.predict(X_test_s)
acc_s = accuracy_score(y_test_s, y_pred_s)
f1_s = f1_score(y_test_s, y_pred_s, average='macro', zero_division=0)

print(f"Multi-Disease Test Accuracy: {acc_s * 100:.2f}%")
print(f"Multi-Disease Macro F1:     {f1_s * 100:.2f}%")

================================================================================
# 🏆 Step 4: Model Comparison & Hackathon Leaderboard
================================================================================
Compare all models side-by-side in a presentation-ready table.

In [ ]:
leaderboard = pd.DataFrame([
    {
        "Task": "Type 2 Diabetes Risk",
        "Architecture": diab_metrics["Model"],
        "Accuracy": f"{diab_metrics['Accuracy']*100:.2f}%",
        "Recall (Sensitivity)": f"{diab_metrics['Recall']*100:.2f}%",
        "ROC-AUC": f"{diab_metrics['ROC-AUC']*100:.2f}%",
        "F1-Score": f"{diab_metrics['F1-Score']*100:.2f}%"
    },
    {
        "Task": "Hypertension Risk",
        "Architecture": htn_metrics["Model"],
        "Accuracy": f"{htn_metrics['Accuracy']*100:.2f}%",
        "Recall (Sensitivity)": f"{htn_metrics['Recall']*100:.2f}%",
        "ROC-AUC": f"{htn_metrics['ROC-AUC']*100:.2f}%",
        "F1-Score": f"{htn_metrics['F1-Score']*100:.2f}%"
    },
    {
        "Task": "General Diagnosis (41 Diseases)",
        "Architecture": "Multi-Class Ensemble (100 Trees across 132 Symptoms)",
        "Accuracy": f"{acc_s*100:.2f}%",
        "Recall (Sensitivity)": f"{f1_s*100:.2f}% (Macro)",
        "ROC-AUC": "N/A (Multi-Class)",
        "F1-Score": f"{f1_s*100:.2f}%"
    }
])

print("\n======================== FINAL HACKATHON MODEL LEADERBOARD ========================")
display(leaderboard)
print("===================================================================================")

================================================================================
# 🧪 Step 5: Interactive Real-Time Prediction Test
================================================================================
Verify live inference on test patient cases across all models.

In [ ]:
import json

# Test Case: Diabetes Hybrid Stacking
sample_diab_patient = pd.DataFrame([{
    'age': 65.0, 'hypertension': 1, 'heart_disease': 1, 'bmi': 34.0,
    'HbA1c_level': 8.2, 'blood_glucose_level': 195.0,
    'gender_Male': 1, 'gender_Other': 0,
    'smoking_history_current': 1, 'smoking_history_ever': 0, 'smoking_history_former': 0,
    'smoking_history_never': 0, 'smoking_history_not current': 0
}])[expected_diab_features]

d_prob = diab_stacking_model.predict_proba(sample_diab_patient)[0][1]
print(f"🧪 Diabetes Stacking Prediction: {'HIGH RISK' if d_prob >= 0.5 else 'LOW RISK'} ({d_prob*100:.1f}% Risk Probability)")

# Test Case: Multi-Disease Symptoms (Pneumonia)
test_symptoms = ['cough', 'high_fever', 'breathlessness', 'chest_pain']
input_vec = np.zeros(len(unique_symptoms))
for s in test_symptoms:
    if s in unique_symptoms:
        input_vec[unique_symptoms.index(s)] = 1

s_probs = multi_disease_model.predict_proba(pd.DataFrame([input_vec], columns=unique_symptoms))[0]
top_indices = np.argsort(s_probs)[::-1][:3]

print(f"\n🧪 Symptom Check for: {test_symptoms}")
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank}. {multi_disease_model.classes_[idx]}: {s_probs[idx]*100:.1f}% confidence")

================================================================================
# 📦 Step 6: Export & One-Click Download Bundle for Backend Integration
================================================================================
Packages all 3 trained models and their JSON metadata into a single zip file and triggers an automatic browser download.

In [ ]:
import joblib
import zipfile

export_dir = Path("hybrid_model_artifacts")
export_dir.mkdir(exist_ok=True)

# 1. Save Diabetes Stacking Model & Metadata
joblib.dump(diab_stacking_model, export_dir / "diabetes_hybrid_stacking_model.pkl", compress=3)
with open(export_dir / "diabetes_features.json", "w") as f:
    json.dump({"features": expected_diab_features}, f, indent=2)

# 2. Save Hypertension Blended Voting Model & Metadata
joblib.dump(htn_voting_model, export_dir / "hypertension_hybrid_voting_model.pkl", compress=3)
with open(export_dir / "hypertension_features.json", "w") as f:
    json.dump({"features": expected_htn_features}, f, indent=2)

# 3. Save Multi-Disease Model & Symptoms/Precautions Encyclopedia
joblib.dump(multi_disease_model, export_dir / "multi_disease_model.pkl", compress=3)
with open(export_dir / "symptoms_list.json", "w") as f:
    json.dump({"total": len(unique_symptoms), "symptoms": unique_symptoms}, f, indent=2)

# Build descriptions and precautions encyclopedia
desc_dict = {str(r['Disease']).strip(): str(r['Description']).strip() for _, r in df_s_desc.iterrows()}
prec_dict = {}
for _, r in df_s_prec.iterrows():
    prec_dict[str(r['Disease']).strip()] = [str(r[c]).strip().capitalize() for c in ['Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4'] if pd.notna(r.get(c)) and str(r.get(c)).strip()]

disease_meta = {}
for d in multi_disease_model.classes_:
    disease_meta[d] = {
        "disease": d,
        "description": desc_dict.get(d, "Clinical condition"),
        "precautions": prec_dict.get(d, ["Consult a healthcare professional"])
    }

with open(export_dir / "disease_metadata.json", "w") as f:
    json.dump(disease_meta, f, indent=2)

# 4. Zip all artifacts
bundle_zip = "family_health_hybrid_models.zip"
with zipfile.ZipFile(bundle_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in export_dir.glob("*"):
        z.write(p, arcname=p.name)

print(f"\n🎁 Created Master Artifacts Bundle: {bundle_zip} ({os.path.getsize(bundle_zip)/1024:.1f} KB)")
print("\n⬇️ Downloading family_health_hybrid_models.zip to your computer now...")
files.download(bundle_zip)
print("\n✅ Done! You can drop these exported models directly into backend/models/")